# 단층 Complex Dynamic BAM (Complex IQ Denoising)

## 조건 요약
- 입력: 복소 IQ 시퀀스, shape `[B, T, 1]`, dtype `torch.cfloat` (또는 numpy complex64)
- 활성화: 실수부/허수부 분리 `tanh` 후 complex 재결합 (split tanh)
- 가중치: 복소수 가중치 `W_uv`, `W_vu`
- 동역학: Euler discretization, inner step 동안 **외부 입력 추가 주입 없음** (자율 동역학)
- 감쇠항 D: 항상 양수 (softplus + floor)
- 학습: supervised (noisy → clean), 복소 MSE(real+imag)
- 검증: `MSE(noisy, GT)` 대비 `MSE(restored, GT)` 감소 확인

## 데이터
- [Model-complex/generate.ipynb](Model-complex/generate.ipynb)에서 clean/noisy를 **미리 저장**
- noisy는 SNR `0 ~ -30 dB` 범위에서 생성되며, 이 노트북에서 SNR을 **5dB 단위로 binning 집계**합니다

## BAM 코드
- BAM 구현은 [utils/LoRa.py](utils/LoRa.py)의 `ComplexIQBAM`을 import해서 사용합니다.

In [ ]:
# ===== Imports / Setup =====
import os
import math
import time
import random
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# utils 모듈 import
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.LoRa import ComplexIQBAM

def complex_mse(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    # complex MSE = MSE(real) + MSE(imag)
    return F.mse_loss(pred.real, target.real) + F.mse_loss(pred.imag, target.imag)

def seed_all(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_all(0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
print('torch:', torch.__version__)

In [ ]:
# ===== Dataset loader (pre-generated clean+noisy pairs) =====
@dataclass
class DataConfig:
    base_dir: str = "dataset_sf9_bw125k_osf8_fs1MHz"
    batch_size: int = 64
    num_workers: int = 0  # macOS: 0 권장
    train_ratio: float = 0.9

cfg = DataConfig()
print(cfg)

base_dir = Path(cfg.base_dir)
clean_dir = base_dir / "clean_iq_norm"
noisy_dir = base_dir / "noisy_iq_norm"
assert clean_dir.exists(), f"Not found: {clean_dir} (먼저 generate.ipynb 실행)"
assert noisy_dir.exists(), f"Not found: {noisy_dir} (먼저 generate.ipynb 실행)"

# noisy 파일명 예: sym_000_rep00_snr-12.3dB_iq.npy
_re_sym = re.compile(r"sym_(\d{3})_")
_re_snr = re.compile(r"snr([+-]\d+(?:\.\d+)?)dB")

class NoisyCleanPairFilesDataset(Dataset):
    def __init__(self, noisy_files: list[Path], clean_dir: Path):
        self.noisy_files = list(noisy_files)
        self.clean_dir = clean_dir

    def __len__(self):
        return len(self.noisy_files)

    def __getitem__(self, idx):
        noisy_path = self.noisy_files[idx]
        m_sym = _re_sym.search(noisy_path.name)
        if m_sym is None:
            raise ValueError(f"Cannot parse sym id from: {noisy_path.name}")
        sym = int(m_sym.group(1))

        m_snr = _re_snr.search(noisy_path.name)
        if m_snr is None:
            raise ValueError(f"Cannot parse snr from: {noisy_path.name}")
        snr_db = float(m_snr.group(1))

        clean_path = self.clean_dir / f"sym_{sym:03d}_iq.npy"
        x_noisy = np.load(noisy_path).astype(np.complex64)  # (T,)
        x_clean = np.load(clean_path).astype(np.complex64)  # (T,)

        noisy_t = torch.from_numpy(x_noisy).to(torch.cfloat).unsqueeze(-1)
        clean_t = torch.from_numpy(x_clean).to(torch.cfloat).unsqueeze(-1)
        snr_t = torch.tensor(snr_db, dtype=torch.float32)
        return noisy_t, clean_t, snr_t

# 파일 리스트
all_noisy = sorted(noisy_dir.glob("sym_*_iq.npy"))
assert len(all_noisy) > 0, f"No noisy files in {noisy_dir}"
print("#noisy files:", len(all_noisy))

# split
rng_split = np.random.default_rng(123)
perm = rng_split.permutation(len(all_noisy))
n_train = int(len(all_noisy) * cfg.train_ratio)
train_noisy = [all_noisy[i] for i in perm[:n_train]]
val_noisy = [all_noisy[i] for i in perm[n_train:]]

train_ds = NoisyCleanPairFilesDataset(train_noisy, clean_dir)
val_ds = NoisyCleanPairFilesDataset(val_noisy, clean_dir)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=(device.type=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=(device.type=='cuda'))

# sanity
noisy_b, clean_b, snr_b = next(iter(train_loader))
print('batch noisy:', noisy_b.shape, noisy_b.dtype)
print('batch clean:', clean_b.shape, clean_b.dtype)
print('batch snr :', snr_b.shape, snr_b.dtype, 'min/max=', float(snr_b.min()), float(snr_b.max()))
assert noisy_b.ndim == 3 and noisy_b.shape[-1] == 1
assert noisy_b.dtype.is_complex
print('T inferred:', noisy_b.shape[1])

In [ ]:
# ===== ComplexIQBAM (imported from utils/LoRa.py) =====
# inject_input=False 로 설정하면 inner step에서 +x_t 주입 없이 자율 동역학으로 수렴
model = ComplexIQBAM(
    m_units=64,
    dt=0.05,
    steps=25,
    decay_init=1.0,
    decay_floor=0.1,
    w_gain_init=0.1,
    stability_margin=0.5,
    use_gain_clamp=True,
    anchor_strength=0.05,
    inject_input=False,
    init_scale=0.02,
).to(device)

noisy_b, clean_b, snr_b = next(iter(train_loader))
noisy_b = noisy_b.to(device)
clean_b = clean_b.to(device)
out_b = model(noisy_b)
print('out:', out_b.shape, out_b.dtype)
print('stability:', model.stability_info())
print('snr sample:', snr_b[:8].tolist())
assert out_b.shape == noisy_b.shape

In [ ]:
# ===== Training / Evaluation (overall + SNR-bin(5dB) stats) =====
def _per_sample_mse(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    # (real^2 + imag^2) mean over (T,1) -> [B]
    e = a - b
    return (e.real**2 + e.imag**2).mean(dim=(1,2))

def snr_to_bin(snr_db: float, step: int = 5) -> int:
    # e.g., -12.3 -> -15, -0.1 -> -5, 0.0 -> 0
    return int(math.floor(snr_db / step) * step)

@torch.no_grad()
def eval_overall_and_bins(model: nn.Module, loader: DataLoader, device: torch.device, *, step_db: int = 5):
    model.eval()
    total_sum_noisy = 0.0
    total_sum_rest = 0.0
    total_count = 0

    # bin -> (sum_noisy, sum_rest, count)
    bins = {}

    for noisy, clean, snr_db in loader:
        noisy = noisy.to(device)
        clean = clean.to(device)
        snr_db = snr_db.detach().cpu().numpy().astype(np.float32)

        restored = model(noisy)

        per_noisy = _per_sample_mse(noisy, clean).detach().cpu().numpy()
        per_rest  = _per_sample_mse(restored, clean).detach().cpu().numpy()

        bsz = per_noisy.shape[0]
        total_sum_noisy += float(per_noisy.sum())
        total_sum_rest  += float(per_rest.sum())
        total_count += int(bsz)

        for i in range(bsz):
            k = snr_to_bin(float(snr_db[i]), step_db)
            if k not in bins:
                bins[k] = [0.0, 0.0, 0]
            bins[k][0] += float(per_noisy[i])
            bins[k][1] += float(per_rest[i])
            bins[k][2] += 1

    overall_noisy = total_sum_noisy / max(1, total_count)
    overall_rest  = total_sum_rest / max(1, total_count)
    return overall_noisy, overall_rest, bins

def _print_bins(bins: dict):
    # bins: {k: [sum_noisy, sum_rest, count]}
    keys = sorted(bins.keys())
    print("SNR_bin(dB) |   N |   MSE(noisy) | MSE(restored) | ratio(noisy/rest)")
    print("-" * 72)
    for k in keys:
        s_noisy, s_rest, n = bins[k]
        if n == 0:
            continue
        m_noisy = s_noisy / n
        m_rest = s_rest / n
        ratio = m_noisy / (m_rest + 1e-12)
        print(f"{k:>10} | {n:>3d} | {m_noisy:>11.6e} | {m_rest:>12.6e} | {ratio:>6.3f}x")

def train_one(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader, device: torch.device, *,
              num_epochs: int = 20, lr: float = 5e-3, weight_decay: float = 0.0, grad_clip: float = 1.0, step_db: int = 5):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # baseline before training
    base_noisy, base_rest, base_bins = eval_overall_and_bins(model, val_loader, device, step_db=step_db)
    print(f"[pre]  val overall MSE(noisy,GT)={base_noisy:.6e} | MSE(restored,GT)={base_rest:.6e}")
    _print_bins(base_bins)

    for epoch in range(1, num_epochs + 1):
        model.train()
        t0 = time.time()
        running = 0.0
        seen = 0

        for noisy, clean, _ in train_loader:
            noisy = noisy.to(device)
            clean = clean.to(device)

            pred = model(noisy)
            loss = complex_mse(pred, clean)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            if grad_clip is not None and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

            bsz = noisy.shape[0]
            running += float(loss.item()) * bsz
            seen += bsz

        train_loss = running / max(1, seen)
        val_noisy, val_rest, _ = eval_overall_and_bins(model, val_loader, device, step_db=step_db)
        dt_s = time.time() - t0
        print(f"Epoch {epoch:03d} | train={train_loss:.6e} | val noisy={val_noisy:.6e} | val restored={val_rest:.6e} | {dt_s:.1f}s")

    # final
    val_noisy, val_rest, val_bins = eval_overall_and_bins(model, val_loader, device, step_db=step_db)
    print(f"[post] val overall MSE(noisy,GT)={val_noisy:.6e} | MSE(restored,GT)={val_rest:.6e} | improvement={val_noisy/(val_rest+1e-12):.3f}x")
    _print_bins(val_bins)

model = ComplexIQBAM(
    m_units=64,
    dt=0.05,
    steps=25,
    decay_init=1.0,
    decay_floor=0.1,
    w_gain_init=0.1,
    stability_margin=0.5,
    use_gain_clamp=True,
    anchor_strength=0.05,
    inject_input=False,
    init_scale=0.02,
).to(device)
print('initial stability:', model.stability_info())

train_one(
    model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    num_epochs=15,
    lr=5e-3,
    weight_decay=0.0,
    grad_clip=1.0,
    step_db=5,
)

In [ ]:
# ===== (Optional) quick qualitative peek on one batch =====
@torch.no_grad()
def summarize_batch(model, loader, device):
    model.eval()
    noisy, clean, snr = next(iter(loader))
    noisy = noisy.to(device)
    clean = clean.to(device)
    restored = model(noisy)

    per_noisy = _per_sample_mse(noisy, clean)
    per_rest  = _per_sample_mse(restored, clean)
    print('snr[:8]              :', snr[:8].tolist())
    print('per-sample MSE noisy :', per_noisy[:8].detach().cpu().numpy())
    print('per-sample MSE rest  :', per_rest[:8].detach().cpu().numpy())
    print('fraction improved:', float((per_rest < per_noisy).float().mean().item()))

summarize_batch(model, val_loader, device)